1. Import Libraries

In [1]:
import pandas as pd
import pyodbc
from pathlib import Path

2. Load CSV Files

In [2]:
#Define the Project Paths
# Project root
PROJECT_ROOT = Path.cwd().parent

# Warehouse folder
WAREHOUSE_PATH = PROJECT_ROOT / "data" / "warehouse"

print(WAREHOUSE_PATH)

c:\Users\محمد الرويلي\OneDrive\Desktop\yaqeen\Retail-Analytics-Project\data\warehouse


In [3]:
#Load the csv files
dim_country = pd.read_csv(WAREHOUSE_PATH / "DimCountry.csv")
dim_customer = pd.read_csv(WAREHOUSE_PATH / "DimCustomer.csv")
dim_date = pd.read_csv(WAREHOUSE_PATH / "DimDate.csv")
dim_product = pd.read_csv(WAREHOUSE_PATH / "DimProduct.csv")
fact_sales = pd.read_csv(WAREHOUSE_PATH / "FactSales.csv")

print("Files loaded successfully.")

Files loaded successfully.


C:\Users\محمد الرويلي\AppData\Local\Temp\ipykernel_6624\3521393952.py:6: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  fact_sales = pd.read_csv(WAREHOUSE_PATH / "FactSales.csv")


In [4]:
#Verify them
print(dim_country.shape)
print(dim_customer.shape)
print(dim_date.shape)
print(dim_product.shape)
print(fact_sales.shape)

(38, 6)
(4339, 19)
(305, 13)
(4161, 6)
(578575, 15)


3. Connect to SQL server

In [5]:
# SQL Server connection
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=RetailAnalyticsDW_v2;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(connection_string)

cursor = conn.cursor()

print("Connected to SQL Server successfully.")

Connected to SQL Server successfully.


In [6]:
#Test the connection
cursor.execute("""
SELECT @@SERVERNAME,
       DB_NAME(),
       @@VERSION
""")

row = cursor.fetchone()

print("Server:", row[0])
print("Database:", row[1])
print("Version:", row[2][:80], "...")

Server: LAPTOP-N3BTN5PO
Database: RetailAnalyticsDW_v2
Version: Microsoft SQL Server 2025 (RTM-GDR) (KB5102333) - 17.0.1125.2 (X64) 
	Jun 18 202 ...


In [7]:
#Enable Fast Bulk Inserts
cursor.fast_executemany = True

print("Fast executemany enabled.")

Fast executemany enabled.


4. Create Tables

In [8]:
#Create the Warehouse Tables
create_tables_sql = """

--====================================================
-- Drop Existing Tables
--====================================================

IF OBJECT_ID('FactSales', 'U') IS NOT NULL DROP TABLE FactSales;
IF OBJECT_ID('DimCustomer', 'U') IS NOT NULL DROP TABLE DimCustomer;
IF OBJECT_ID('DimProduct', 'U') IS NOT NULL DROP TABLE DimProduct;
IF OBJECT_ID('DimCountry', 'U') IS NOT NULL DROP TABLE DimCountry;
IF OBJECT_ID('DimDate', 'U') IS NOT NULL DROP TABLE DimDate;

--====================================================
-- DimDate
--====================================================

CREATE TABLE DimDate(

    DateKey INT PRIMARY KEY,

    FullDate DATE,

    Year INT,

    Quarter INT,

    Month INT,

    MonthName NVARCHAR(20),

    Week INT,

    Day INT,

    DayName NVARCHAR(20),

    IsWeekend BIT,

    IsMonthEnd BIT,

    IsQuarterEnd BIT,

    IsYearEnd BIT

);

--====================================================
-- DimCustomer
--====================================================

CREATE TABLE DimCustomer(

    CustomerKey INT PRIMARY KEY,

    CustomerID INT,

    Country NVARCHAR(100),

    FirstPurchaseDate DATE,

    LastPurchaseDate DATE,

    CustomerAgeDays INT,

    TotalOrders INT,

    TotalSales DECIMAL(18,2),

    TotalQuantity INT,

    AverageOrderValue DECIMAL(18,2),

    Recency INT,

    Frequency INT,

    Monetary DECIMAL(18,2),

    R_Score INT,

    F_Score INT,

    M_Score INT,

    RFMScore INT,

    CustomerSegment NVARCHAR(50),

    LoyaltyLevel NVARCHAR(50)

);

--====================================================
-- DimProduct
--====================================================

CREATE TABLE DimProduct (
    ProductKey INT PRIMARY KEY,

    StockCode NVARCHAR(50),

    Description NVARCHAR(255),

    Department NVARCHAR(100),

    Category NVARCHAR(100),

    Subcategory NVARCHAR(100)
);

--====================================================
-- DimCountry
--====================================================

CREATE TABLE DimCountry(

    CountryKey INT PRIMARY KEY,

    Country NVARCHAR(100),

    Region NVARCHAR(100),

    Continent NVARCHAR(100),

    Market NVARCHAR(100),

    Currency NVARCHAR(50)

);

--====================================================
-- FactSales
--====================================================

CREATE TABLE FactSales(

    SalesKey INT PRIMARY KEY,

    InvoiceNo NVARCHAR(50),

    DateKey INT,

    CustomerKey INT,

    ProductKey INT,

    CountryKey INT,

    Quantity INT,

    UnitPrice DECIMAL(18,2),

    DiscountPercent INT,

    SalesAmount DECIMAL(18,2),

    CostRatio DECIMAL(5,2),

    UnitCost DECIMAL(18,2),

    TotalCost DECIMAL(18,2),

    Profit DECIMAL(18,2),

    ProfitMargin DECIMAL(18,2)

);
"""

In [9]:
#Execute the SQL Script
cursor.execute(create_tables_sql)

conn.commit()

print("Warehouse tables created successfully.")

Warehouse tables created successfully.


In [10]:
#Verify the Tables
cursor.execute("""
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE='BASE TABLE'
ORDER BY TABLE_NAME
""")

for row in cursor.fetchall():
    print(row[0])

DimCountry
DimCustomer
DimDate
DimProduct
FactSales


5. Insert Data

In [11]:
#the new function ()
def load_table(df, table_name):
    """
    Load a DataFrame into SQL Server using pyodbc.
    Uses normal executemany for DimCustomer and fast_executemany for all other tables.
    """

    cursor = conn.cursor()

    # Disable fast_executemany only for DimCustomer
    if table_name == "DimCustomer":
        cursor.fast_executemany = False
    else:
        cursor.fast_executemany = True

    # Build INSERT statement
    columns = ", ".join(df.columns)
    placeholders = ", ".join(["?"] * len(df.columns))

    sql = f"""
    INSERT INTO dbo.{table_name}
    ({columns})
    VALUES ({placeholders})
    """

    # Convert DataFrame to tuples
    rows = list(df.itertuples(index=False, name=None))

    # Insert rows
    cursor.executemany(sql, rows)

    # Commit
    conn.commit()

    print(f"✓ {table_name}: {len(df):,} rows inserted.")

    cursor.close()

In [12]:
# Clear tables 2
cursor = conn.cursor()

cursor.execute("DELETE FROM FactSales")
cursor.execute("DELETE FROM DimProduct")
cursor.execute("DELETE FROM DimCustomer")
cursor.execute("DELETE FROM DimDate")
cursor.execute("DELETE FROM DimCountry")

conn.commit()
cursor.close()

In [13]:
load_table(dim_country, "DimCountry")

✓ DimCountry: 38 rows inserted.


In [14]:
load_table(dim_date, "DimDate")

✓ DimDate: 305 rows inserted.


In [15]:
load_table(dim_customer, "DimCustomer")

✓ DimCustomer: 4,339 rows inserted.


In [16]:
load_table(dim_product, "DimProduct")

✓ DimProduct: 4,161 rows inserted.


In [17]:
print(dim_product.columns.tolist())

['ProductKey', 'StockCode', 'Description', 'Department', 'Category', 'Subcategory']


In [18]:
load_table(fact_sales, "FactSales")

✓ FactSales: 578,575 rows inserted.
